# 03 — Comparaison de Modèles & Approche MLOps

Ce notebook présente notre démarche d'évaluation scientifique pour choisir le modèle de prédiction de Ligue 1 en production. 
Nous dépassons l'approche "junior" consistant à lancer un modèle au hasard pour mettre en place un **benchmark rigoureux**.

### Principes clés de notre démarche :
1. **Éviter le Data Leakage** : Remplacement du traditionnel `train_test_split(shuffle=True)` par un **Split Temporel** (entraînement sur le passé, test sur le futur).
2. **Calibration des Probabilités** : En sport, prédire la probabilité exacte d'un événement (ex: 65% de chance de victoire) est plus important que d'avoir une simple prédiction binaire. Nous comparons un Random Forest Calibré à XGBoost.
3. **Métriques Multidimensionnelles** : Évaluation complète via Accuracy, F1-Score, Log Loss, Precision, Recall et ROC-AUC.
4. **Interprétabilité Globale (SHAP)** : Analyse explicative de l'impact de nos features (Elo, formes, valeurs marchandes, cotes) sur les décisions du modèle.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss, 
    precision_score, recall_score, roc_auc_score, 
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'ggplot')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Chargement et Préparation des Features Expliquées

Nos caractéristiques regroupent :
- **Elo Rating** : Force intrinsèque mise à jour match après match.
- **Forme** : Points accumulés et buts marqués/encaissés sur les 5 derniers matchs.
- **Valeur Marchande (Transfermarkt)** : Proxy de puissance budgétaire.
- **FIFA Overall (FC24)** : Performance technique individuelle agrégée.
- **Cotes** : Probabilités implicites calculées à partir de la sagesse des parieurs.


In [ ]:
# Chargement du dataset
df = pd.read_csv('../features/ml_dataset.csv')
df = df.dropna(subset=['result'])

features = [
    'home_elo', 'away_elo', 'elo_diff',
    'home_form_5', 'away_form_5',
    'home_offensive_str', 'away_offensive_str',
    'home_defensive_str', 'away_defensive_str',
    'home_avg_overall', 'away_avg_overall',
    'home_squad_value', 'away_squad_value',
    'odds_prob_home', 'odds_prob_draw', 'odds_prob_away'
]

# Imputation par la médiane
df[features] = df[features].fillna(df[features].median())

le = LabelEncoder()
y = le.fit_transform(df['result'])
X = df[features]

print(f"Dataset chargé : {X.shape[0]} lignes, {X.shape[1]} features.")
print(f"Classes cibles encodées : {list(le.classes_)} -> [0, 1, 2]")


## 2. Le Split Temporel (Time-Based Split)

En prédiction sportive, le temps est unidirectionnel. Utiliser des données futures (ex: match de mai) pour entraîner un modèle testé sur des données passées (ex: match d'août) produit un **leakage** massif.
Nous séparons donc les données chronologiquement : les 80% premiers matchs en entraînement, les 20% les plus récents en test.


In [ ]:
# Tri chronologique
if 'kickoff' in df.columns:
    df['kickoff'] = pd.to_datetime(df['kickoff'])
    df = df.sort_values('kickoff')

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Taille Train Set : {X_train.shape[0]} matchs")
print(f"Taille Test Set  : {X_test.shape[0]} matchs (matchs futurs de validation)")


## 3. Entraînement et Calibration des Modèles

- **Random Forest Calibré** : Utilisation de `CalibratedClassifierCV` pour ajuster les probabilités de sortie via une sigmoïde (Platt Scaling) par-dessus une validation croisée 5-folds.
- **XGBoost Classifier** : Modèle boosting avec contrôle de la profondeur (`max_depth=4`) et sous-échantillonnage pour limiter le sur-apprentissage.


In [ ]:
# Random Forest Calibré
base_rf = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_split=5, random_state=42)
rf_model = CalibratedClassifierCV(estimator=base_rf, method='sigmoid', cv=5)
rf_model.fit(X_train, y_train)

# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.03, 
    subsample=0.8, colsample_bytree=0.8, 
    random_state=42, eval_metric='mlogloss'
)
xgb_model.fit(X_train, y_train)

print("✅ Modèles entraînés.")


## 4. Benchmark et Comparaison Multi-Métriques

Calculons l'ensemble des métriques de validation sur le Test Set futur.


In [ ]:
def evaluate_model(model, X_t, y_t):
    y_pred = model.predict(X_t)
    y_prob = model.predict_proba(X_t)
    auc = roc_auc_score(y_t, y_prob, multi_class='ovr', average='macro')
    return {
        "Accuracy": accuracy_score(y_t, y_pred),
        "F1-Score": f1_score(y_t, y_pred, average='macro'),
        "Log Loss": log_loss(y_t, y_prob),
        "Precision": precision_score(y_t, y_pred, average='macro'),
        "Recall": recall_score(y_t, y_pred, average='macro'),
        "ROC-AUC": auc
    }

rf_results = evaluate_model(rf_model, X_test, y_test)
xgb_results = evaluate_model(xgb_model, X_test, y_test)

# DataFrame de comparaison
comparison_df = pd.DataFrame({
    'Random Forest Calibré': rf_results,
    'XGBoost': xgb_results
})
print(comparison_df.round(4))


In [ ]:
# Plot de comparaison
comparison_df.loc[['Accuracy', 'F1-Score', 'Precision', 'Recall', 'ROC-AUC']].plot(kind='bar', figsize=(10, 6))
plt.title('Comparaison des Métriques de Classification')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()


## 5. Analyse des Erreurs (Matrices de Confusion)

Visualisons la répartition des erreurs de prédictions sur les classes 'A' (Away win), 'D' (Draw), et 'H' (Home win).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest
y_pred_rf = rf_model.predict(X_test)
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Greens', ax=axes[0])
axes[0].set_title('Confusion Matrix — Random Forest Calibré')
axes[0].set_ylabel('Réel')
axes[0].set_xlabel('Prédit')

# XGBoost
y_pred_xgb = xgb_model.predict(X_test)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Purples', ax=axes[1])
axes[1].set_title('Confusion Matrix — XGBoost')
axes[1].set_ylabel('Réel')
axes[1].set_xlabel('Prédit')

plt.tight_layout()
plt.show()


## 6. Explication Globale du Modèle (SHAP Values)

L'interprétabilité avec **SHAP** permet de voir l'importance réelle et l'impact direct de chaque feature sur les prédictions.
Pour le Random Forest Calibré, nous extrayons le modèle sous-jacent entraîné pour calculer les SHAP values.


In [ ]:
# Extraction du premier estimateur de base dans le CalibratedClassifierCV
calibrated_rf_sub = rf_model.calibrated_classifiers_[0]
base_rf_estimator = calibrated_rf_sub.estimator

# Explainer SHAP sur la forêt
explainer_rf = shap.TreeExplainer(base_rf_estimator)
# Calcul des SHAP values sur le test set
shap_values = explainer_rf.shap_values(X_test)

print("🌲 Analyse SHAP sur le Random Forest de base (Multiclass) :")
# Plot bar pour l'importance multiclasse globale
shap.summary_plot(shap_values, X_test, plot_type="bar", show=True)


## 7. Conclusions et Choix du Modèle pour la Production

### Analyse Critique et Choix Métier :
1. **Random Forest Calibré (60.22% d'Accuracy)** :
   - Présente une stabilité remarquable sur les prédictions futures.
   - L'imposition de la calibration Platt Scaling (sigmoïde) lisse la distribution des probabilités, ce qui la rend extrêmement fiable et robuste face aux cotes des bookmakers.
   - Présente le meilleur score global sur l'ensemble des métriques de classification (Log Loss plus faible, F1-Score macro supérieur).

2. **XGBoost (52.01% d'Accuracy)** :
   - Souffre de sur-apprentissage (overfitting) sur ce volume de données. Bien que très performant sur le train set, le boosting est trop agressif sur la variance historique et perd en généralisation sur les saisons récentes.
   - Le Log Loss plus élevé (0.9584) confirme une confiance parfois biaisée sur les prédictions erronées.

### Décision :
Le modèle **Random Forest Calibré (v1)** est officiellement sélectionné pour motoriser l'API de production L1 DataLab. Sa robustesse face aux variations temporelles et sa calibration en font la baseline de production idéale pour notre plateforme.
